In [1]:
import pandas as pd
from pathlib import Path

proyecto = Path(
    r"C:\Users\Juan Palacios\Documents\Universidad"
    r"\1_Semestre_Especializacion\GESTION_DE_DATOS"
    r"\Proyecto_GYA\ProyectoFinalETL"
)

archivo_men = (
    proyecto / "data" / "bronze"
    / "MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR__B_SICA_Y_MEDIA_POR_MUNICIPIO.csv"
)

men_original = pd.read_csv(
    archivo_men,
    sep=",",
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
    na_values=[""],
)

# Copia de trabajo: conservar men_original para consultar la fuente
men_revision = men_original.apply(
    lambda columna: columna.str.strip().replace("", pd.NA)
)

print("Filas:", men_original.shape[0])
print("Columnas:", men_original.shape[1])

display(men_original.head())

Filas: 15707
Columnas: 41


,AÑO,CÓDIGO_MUNICIPIO,MUNICIPIO,CÓDIGO_DEPARTAMENTO,DEPARTAMENTO,CÓDIGO_ETC,ETC,POBLACIÓN_5_16,TASA_MATRICULACIÓN_5_16,COBERTURA_NETA,...,REPROBACIÓN,REPROBACIÓN_TRANSICIÓN,REPROBACIÓN_PRIMARIA,REPROBACIÓN_SECUNDARIA,REPROBACIÓN_MEDIA,REPITENCIA,REPITENCIA_TRANSICIÓN,REPITENCIA_PRIMARIA,REPITENCIA_SECUNDARIA,REPITENCIA_MEDIA
0,2023,05001,Medellín,05,Antioquia,3759,Medellín,377562,96.15,95.94,...,9.23,0.15,6.21,15.4,7.27,9.25,1.82,8.34,13.69,4.48
1,2023,05002,Abejorral,05,Antioquia,3758,Antioquia (ETC),3634,74.38,74.38,...,7.77,0.56,7.26,10.93,4.41,9.07,1.67,10.28,11.32,2.2
2,2023,05004,Abriaquí,05,Antioquia,3758,Antioquia (ETC),503,62.62,62.62,...,6.55,0,1.96,16.51,2.04,9.52,0,10.46,13.76,2.04
3,2023,05021,Alejandría,05,Antioquia,3758,Antioquia (ETC),864,81.37,81.37,...,7.88,0,7.02,13.65,2.4,7.2,1.59,7.69,11.65,0
4,2023,05030,Amagá,05,Antioquia,3758,Antioquia (ETC),5060,78.3,78.3,...,10.71,0,9.63,16.48,5.5,12.08,0.29,11.54,18.48,3.98


In [2]:
men_revision.info()

calidad_men = pd.DataFrame({
    "faltantes": men_revision.isna().sum(),
    "valores_distintos": men_revision.nunique(),
})

calidad_men["porcentaje_faltante"] = (
    calidad_men["faltantes"] / len(men_revision) * 100
).round(2)

display(
    calidad_men.sort_values(
        "porcentaje_faltante",
        ascending=False,
    )
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15707 entries, 0 to 15706
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   AÑO                          15707 non-null  string
 1   CÓDIGO_MUNICIPIO             15707 non-null  string
 2   MUNICIPIO                    15707 non-null  string
 3   CÓDIGO_DEPARTAMENTO          15707 non-null  string
 4   DEPARTAMENTO                 15707 non-null  string
 5   CÓDIGO_ETC                   15707 non-null  string
 6   ETC                          15707 non-null  string
 7   POBLACIÓN_5_16               15701 non-null  string
 8   TASA_MATRICULACIÓN_5_16      15592 non-null  string
 9   COBERTURA_NETA               15596 non-null  string
 10  COBERTURA_NETA_TRANSICIÓN    15655 non-null  string
 11  COBERTURA_NETA_PRIMARIA      15616 non-null  string
 12  COBERTURA_NETA_SECUNDARIA    15613 non-null  string
 13  COBERTURA_NETA_MEDIA         15

,faltantes,valores_distintos,porcentaje_faltante
TAMAÑO_PROMEDIO_DE_GRUPO,8135,4232,51.79
SEDES_CONECTADAS_A_INTERNET,7939,1402,50.54
DESERCIÓN_TRANSICIÓN,903,1179,5.75
DESERCIÓN_MEDIA,734,1206,4.67
DESERCIÓN_SECUNDARIA,270,1433,1.72
DESERCIÓN_PRIMARIA,242,1004,1.54
REPITENCIA_TRANSICIÓN,159,1010,1.01
REPITENCIA_SECUNDARIA,152,1883,0.97
REPITENCIA_PRIMARIA,148,1606,0.94
REPROBACIÓN_MEDIA,145,1617,0.92


In [3]:
columnas_texto = [
    "CÓDIGO_MUNICIPIO",
    "MUNICIPIO",
    "CÓDIGO_DEPARTAMENTO",
    "DEPARTAMENTO",
    "CÓDIGO_ETC",
    "ETC",
]

columnas_numericas = [
    columna
    for columna in men_revision.columns
    if columna not in columnas_texto
]

men_numericos = men_revision[columnas_numericas].apply(
    lambda columna: pd.to_numeric(
        columna,
        errors="coerce",
    )
)

errores_conversion = (
    men_numericos.isna()
    & men_revision[columnas_numericas].notna()
)

display(
    errores_conversion.sum()
    .rename("valores_no_convertibles")
    .to_frame()
)

# Mostrar los valores originales que fallaron
for columna in columnas_numericas:
    if errores_conversion[columna].any():
        print("Revisar columna:", columna)

        display(
            men_original.loc[
                errores_conversion[columna],
                ["AÑO", "CÓDIGO_MUNICIPIO", "MUNICIPIO", columna],
            ]
        )

,valores_no_convertibles
AÑO,0
POBLACIÓN_5_16,1
TASA_MATRICULACIÓN_5_16,0
COBERTURA_NETA,0
COBERTURA_NETA_TRANSICIÓN,0
COBERTURA_NETA_PRIMARIA,0
COBERTURA_NETA_SECUNDARIA,0
COBERTURA_NETA_MEDIA,0
COBERTURA_BRUTA,0
COBERTURA_BRUTA_TRANSICIÓN,0


Revisar columna: POBLACIÓN_5_16


,AÑO,CÓDIGO_MUNICIPIO,MUNICIPIO,POBLACIÓN_5_16
12282,2021,11001,"Bogotá, D.C.","1,174,274"


In [4]:
disponibilidad_anual = (
    men_revision.assign(
        sedes_internet_faltante=(
            men_revision["SEDES_CONECTADAS_A_INTERNET"].isna()
        ),
        grupo_faltante=(
            men_revision["TAMAÑO_PROMEDIO_DE_GRUPO"].isna()
        ),
    )
    .groupby("AÑO")
    .agg(
        registros=("AÑO", "size"),
        faltantes_sedes_internet=("sedes_internet_faltante", "sum"),
        faltantes_tamano_grupo=("grupo_faltante", "sum"),
    )
    .sort_index()
)

disponibilidad_anual["porcentaje_faltante_sedes"] = (
    disponibilidad_anual["faltantes_sedes_internet"]
    / disponibilidad_anual["registros"]
    * 100
).round(2)

display(disponibilidad_anual)

,registros,faltantes_sedes_internet,faltantes_tamano_grupo,porcentaje_faltante_sedes
AÑO,,,,
2011,1122,5,15,0.45
2012,1122,5,15,0.45
2013,1122,3,190,0.27
2014,1122,4,16,0.36
2015,1122,2,15,0.18
2016,1122,27,16,2.41
2017,1122,40,15,3.57
2018,1122,1122,1122,100.00
2019,1123,1123,1123,100.00


In [5]:
codigo_municipio_normalizado = (
    men_revision["CÓDIGO_MUNICIPIO"]
    .str.zfill(5)
)

clave_men = pd.DataFrame({
    "anio": men_revision["AÑO"],
    "codigo_municipio": codigo_municipio_normalizado,
})

print(
    "Filas completamente duplicadas:",
    men_original.duplicated().sum(),
)

print(
    "Repeticiones de código territorial y año:",
    clave_men.duplicated().sum(),
)

print(
    "Filas con código nacional:",
    codigo_municipio_normalizado.eq("00000").sum(),
)

Filas completamente duplicadas: 0
Repeticiones de código territorial y año: 0
Filas con código nacional: 3


In [6]:
rangos_men = men_numericos.agg(
    ["count", "min", "max"]
).T

rangos_men["valores_negativos"] = (
    men_numericos.lt(0).sum()
)

display(rangos_men)

,count,min,max,valores_negativos
AÑO,15707.0,2011.0,2024.00,0
POBLACIÓN_5_16,15700.0,1.0,9548263.00,0
TASA_MATRICULACIÓN_5_16,15592.0,0.0,279.03,0
COBERTURA_NETA,15596.0,0.0,264.54,0
COBERTURA_NETA_TRANSICIÓN,15655.0,0.0,150.47,0
COBERTURA_NETA_PRIMARIA,15616.0,0.0,254.96,0
COBERTURA_NETA_SECUNDARIA,15613.0,0.0,229.44,0
COBERTURA_NETA_MEDIA,15614.0,0.0,170.26,0
COBERTURA_BRUTA,15639.0,0.0,10448.00,0
COBERTURA_BRUTA_TRANSICIÓN,15610.0,0.0,259.29,0
